# RealMLP — 2-seed ensemble on T4 x2, fe_v4_native

Re-runs RealMLP for stacking, upgrading the baseline run `20260603-021557-d716c1`
(realmlp-baseline) in two ways:

1. **Features:** `fe_v4_native` — the minimal engineered set the GBDTs scored best with
   (`contract_x_internet`, `contract_x_payment`, `AverageMonthly`) on **native
   category-dtype** categoricals instead of fe_v0 one-hot. pytabkit consumes category
   columns directly (RealMLP embeds them internally), and the same upgrade moved
   TabICL's OOF ROC-AUC by +0.0018 (run `20260612-031158-5adf7a`).
2. **Both T4s:** RealMLP trains on a single device and is training-bound, so the second
   GPU is converted into *quality* rather than speed: one model per GPU is trained
   **concurrently** with different seeds (42/43) and their predictions averaged — a
   2-seed deep ensemble at roughly the wall time of a single model.

No subsample bagging here — RealMLP trains on the full fold-training partition, so the
OOF is full-coverage by construction.

**Fold contract.** The outer splitter is the project-wide
`StratifiedKFold(n_splits=5, shuffle=True, random_state=42)`. A check cell asserts the
on-platform `(id, fold)` assignment matches the committed
`experiments/cv_folds_seed42.csv.gz`, so the OOF is guaranteed stackable against the
local runs. **Push that file to GitHub master before running** — the clone must contain it.

**Notebook settings (right sidebar).** Accelerator → **GPU T4 x2** (not P100);
Internet → **On**; Add Input → **playground-series-s6e3**.

> ⏱️ The baseline took **~4 h** for 5 folds on one T4 — this is the long run of the
> family; with seed-pair CPU contention budget **~4–5.5 h** and start it early in a
> fresh GPU session. **Smoke-test first**: run the run-cell on a small slice (see its
> comment), confirm BOTH GPUs light up in `nvidia-smi`, then run the full version. If
> concurrent fits misbehave (pytabkit is not documented as thread-safe), set
> `DEVICES = ('cuda:0',)` in the config cell — that falls back to the baseline's
> single-GPU, single-seed behavior.

In [1]:
# List attached inputs (confirm the competition data is mounted).
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/competitions/playground-series-s6e3/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e3/train.csv
/kaggle/input/competitions/playground-series-s6e3/test.csv


In [2]:
import os, sys, subprocess

REPO_URL  = "https://github.com/biswajit-nag/Predict-Customer-Churn.git"
REPO_ROOT = "/kaggle/working/Predict-Customer-Churn"

if not os.path.exists(REPO_ROOT):
    subprocess.run(["git", "clone", REPO_URL, REPO_ROOT], check=True)

os.chdir(REPO_ROOT)                       # CWD = repo root (fixes data paths + git_info)
# Put the clone FIRST on sys.path so its `src` wins over any other module named `src`,
# and drop a possibly-stale `src` cached by an earlier cell. (A plain
# `if REPO_ROOT not in sys.path` guard can leave a shadowing `src` ahead of ours.)
sys.path.insert(0, REPO_ROOT)
for _m in [k for k in list(sys.modules) if k == "src" or k.startswith("src.")]:
    del sys.modules[_m]
print("CWD:", os.getcwd())

Cloning into '/kaggle/working/Predict-Customer-Churn'...
Updating files: 100% (354/354), done.


CWD: /kaggle/working/Predict-Customer-Churn


In [3]:
!pip install -q pytabkit      # Kaggle's GPU image already ships torch + CUDA

import torch
print("CUDA available:", torch.cuda.is_available())
print("GPUs:", torch.cuda.device_count())     # expect 2 on T4 x2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 364.0/364.0 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 107.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu1

In [4]:
# cuda.is_available() can return True on an incompatible GPU — run a real op on BOTH
# devices (the seed pair trains one model per device, so both must be healthy).
import torch
for d in range(torch.cuda.device_count()):
    try:
        _ = (torch.randn(16, device=f"cuda:{d}") @ torch.randn(16, 16, device=f"cuda:{d}")).sum().item()
        print(f"cuda:{d} ({torch.cuda.get_device_name(d)}): compute OK")
    except Exception as e:
        print(f"cuda:{d} compute FAILED:", e)   # if this fails, switch to T4 x2 and restart

from pytabkit import RealMLP_TD_Classifier
print("RealMLP_TD_Classifier imported OK")

cuda:0 (Tesla T4): compute OK
cuda:1 (Tesla T4): compute OK
RealMLP_TD_Classifier imported OK


In [5]:
# data/processed/*.parquet are git-ignored, so absent from the clone. Rebuild the
# NATIVE (category-dtype) frames from the attached competition CSVs — pytabkit consumes
# category columns directly, so this is the same data path as the GBDT fe_v4 runs.
import shutil
from pathlib import Path

raw_dir = Path(REPO_ROOT) / "data" / "raw"
raw_dir.mkdir(parents=True, exist_ok=True)
for f in ("train.csv", "test.csv"):
    shutil.copy(f"/kaggle/input/competitions/playground-series-s6e3/{f}", raw_dir / f)

from src.data import prepare_data
train_df, test_df = prepare_data(encoding='native', force=True)
print(f'Loaded native: train_df {train_df.shape}, test_df {test_df.shape}')

Preprocessed and saved (native): train_df (594194, 21), test_df (254655, 20)
Loaded native: train_df (594194, 21), test_df (254655, 20)


### Feature engineering — fe_v4_native (minimal set)

Same `engineer_features` as the GBDT min3 runs (`lgbm-catreg-fe-min3`,
`catboost-gpu-fe-min3`): `AverageMonthly` plus the two low-cardinality crosses, all
row-wise (stateless) so there is no leakage when applied before the CV split. No
`tenure == 0` rows exist in the data, so `AverageMonthly` is always finite — relevant
here because pytabkit's preprocessing rejects infinities (the GBDTs would have
tolerated them).

In [6]:
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

from src.tracking import DATA_DIR

DATA_VERSION = 'fe_v4_native'   # minimal engineered set on the native categorical base


def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """Row-wise (stateless) feature engineering — identical to the GBDT min3 runs."""
    df = df.copy()
    df['AverageMonthly'] = df['TotalCharges'] / df['tenure']
    df['contract_x_payment'] = (df['Contract'].astype(str) + ' | '
                                + df['PaymentMethod'].astype(str)).astype('category')
    df['contract_x_internet'] = (df['Contract'].astype(str) + ' | '
                                 + df['InternetService'].astype(str)).astype('category')
    return df


# Cache engineered parquets per DATA_VERSION (same convention as Experiments.ipynb).
fe_train_path = DATA_DIR / f'train_df_{DATA_VERSION}.parquet'
fe_test_path  = DATA_DIR / f'test_df_{DATA_VERSION}.parquet'

if fe_train_path.exists() and fe_test_path.exists():
    train_df = pd.read_parquet(fe_train_path)
    test_df  = pd.read_parquet(fe_test_path)
    print(f'Loaded cached FE: {DATA_VERSION}')
else:
    train_df = engineer_features(train_df)
    test_df  = engineer_features(test_df)
    pq.write_table(pa.Table.from_pandas(train_df, preserve_index=False), fe_train_path)
    pq.write_table(pa.Table.from_pandas(test_df,  preserve_index=False), fe_test_path)
    print(f'Computed and cached FE: {DATA_VERSION}')

Loaded cached FE: fe_v4_native


In [7]:
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score

from src.tracking import RUNS_DIR
from src.cv import run_cv_experiment, save_experiment

encoded_features = [c for c in train_df.columns if c not in ('id', 'Churn')]
X_train = train_df[encoded_features]
y_train = train_df['Churn']
X_test  = test_df[encoded_features]

# pytabkit detects category-dtype columns and encodes them BY VALUE internally
# (RealMLP uses categorical embeddings), so differing train/test category sets cannot
# cause a code mismatch.
cat_features = [c for c in encoded_features if str(X_train[c].dtype) == 'category']
print(f'X_train: {X_train.shape}  X_test: {X_test.shape}  '
      f'features: {len(encoded_features)}  categorical: {len(cat_features)}')

X_train: (594194, 22)  X_test: (254655, 22)  features: 22  categorical: 17


### Fold-contract check

Stacking aligns OOF vectors by row position across runs from different environments, so
this cell asserts the contract instead of assuming it: the on-platform regenerated rows
(by `id`) and the `StratifiedKFold(5, shuffle, seed 42)` fold assignment must exactly
match `experiments/cv_folds_seed42.csv.gz`, committed from the local environment by
`scripts/make_cv_folds.py`. If either assert fires, **stop** — do not save the run.

In [8]:
folds_ref = pd.read_csv('experiments/cv_folds_seed42.csv.gz')   # CWD = repo root

cv_check = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
folds_here = np.full(len(train_df), -1)
# X is only consulted for its length; zeros keep this independent of features.
for fold, (_, va_idx) in enumerate(cv_check.split(np.zeros(len(train_df)), train_df['Churn'])):
    folds_here[va_idx] = fold

assert (folds_ref['id'].to_numpy() == train_df['id'].to_numpy()).all(), \
    'Row order differs from local — OOF would NOT be stackable. Stop.'
assert (folds_ref['fold'].to_numpy() == folds_here).all(), \
    'Fold assignment differs from local — OOF would NOT be stackable. Stop.'
print('Fold contract OK: rows and folds match the committed local assignment.')

Fold contract OK: rows and folds match the committed local assignment.


### 2-seed ensemble wrapper (one model per GPU)

RealMLP is training-bound and single-device, so unlike the TabPFN/TabICL wrappers
(which replicate models and split *prediction* rows), this wrapper trains **one
independent model per device, concurrently** — seed `random_state + i` on `devices[i]`
— and averages their `predict_proba`. Both models see the full fold-training partition
(no subsampling), so the OOF stays full-coverage and leakage-free exactly as in the
baseline run; the seed pair only averages over initialization/data-order noise. Torch
releases the GIL during GPU work, so two fits in threads overlap almost fully; the
shared-CPU preprocessing adds some contention.

Subclassing `BaseEstimator` makes `get_params()` return the clean constructor args, so
the run's `params.json` and `params_hash` stay informative.

In [9]:
from concurrent.futures import ThreadPoolExecutor

from sklearn.base import BaseEstimator, ClassifierMixin


class SeedEnsembleRealMLP(BaseEstimator, ClassifierMixin):
    """RealMLP 2-seed deep ensemble, one model per GPU (see markdown above)."""

    def __init__(self, n_cv=1, n_refit=0, random_state=42,
                 devices=('cuda:0', 'cuda:1')):
        self.n_cv         = n_cv
        self.n_refit      = n_refit
        self.random_state = random_state
        self.devices      = devices

    def fit(self, X, y):
        X = X.reset_index(drop=True)
        y = y.reset_index(drop=True)

        def fit_one(arg):
            i, dev = arg
            model = RealMLP_TD_Classifier(
                device=dev,
                random_state=self.random_state + i,   # independent seed per device
                n_cv=self.n_cv,        # no internal CV ensembling — one model per fit
                n_refit=self.n_refit,
            )
            model.fit(X, y)
            return model

        with ThreadPoolExecutor(max_workers=len(self.devices)) as pool:
            self.models_ = list(pool.map(fit_one, enumerate(self.devices)))
        self.classes_ = self.models_[0].classes_
        return self

    def predict_proba(self, X):
        with ThreadPoolExecutor(max_workers=len(self.devices)) as pool:
            probas = list(pool.map(lambda m: m.predict_proba(X), self.models_))
        return np.mean(probas, axis=0)

### Run configuration

Same `run_config` shape as the baseline; `metric=accuracy_score` mirrors the other
runs and the harness always logs OOF ROC-AUC (the project's primary metric)
separately. `save_models=False` because RealMLP is torch-backed (fragile to
`joblib.dump`).

In [10]:
from importlib.metadata import version

# Fall back to ('cuda:0',) here if concurrent fits misbehave (see header).
DEVICES = tuple(f'cuda:{i}' for i in range(max(1, torch.cuda.device_count())))

run_config = {
    'model_factory': lambda params: SeedEnsembleRealMLP(**params),
    'params': {
        'n_cv':         1,    # no internal CV ensembling — one model per outer fold
        'n_refit':      0,
        'random_state': 42,
        'devices':      DEVICES,
    },
    'metric':        accuracy_score,
    'metric_name':   'accuracy',
    'cv':            StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    'tag':           'realmlp-fe-min3',
    'notes': (
        'RealMLP (pytabkit RealMLP_TD_Classifier, tuned defaults) 2-seed deep ensemble: '
        'one model per T4 trained concurrently (seeds 42/43), predictions averaged; full '
        'fold-training partition per model (no subsampling). FE upgraded from the parent '
        'run\u2019s fe_v0 one-hot to fe_v4_native (contract_x_internet + contract_x_payment '
        '+ AverageMonthly, native category dtype consumed directly). Fold assignment '
        'asserted against committed experiments/cv_folds_seed42.csv.gz. '
        f'pytabkit={version("pytabkit")}, torch={version("torch")}. '
        'Data regenerated on-platform; GPU run not bit-reproducible. '
        'Notebook: kaggle/predict-customer-churn-realmlp-gpu-min3.ipynb.'
    ),
    'parent_run_id': '20260603-021557-d716c1',
    'save_models':   False,
    'data_version':  DATA_VERSION,
}

In [11]:
# Step 1 — Run the experiment. SMOKE-TEST FIRST: run once on a slice to confirm both
# GPUs show load in nvidia-smi and the concurrent fits behave, then discard and run full:
#   result = run_cv_experiment(run_config, X_train.head(20_000), y_train.head(20_000),
#                              X_test.head(10_000), encoded_features)
# Keep n_splits=5 for the real run — the fold contract requires it.
result = run_cv_experiment(run_config, X_train, y_train, X_test, encoded_features)

Run ID: 20260612-185205-91e530
Tag:    realmlp-fe-min3



/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
GPU available: True (cuda), used: True
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud

Fold 0: accuracy=0.8584  roc_auc=0.9137  (fit 5959.4s)


/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
GPU available: True (cuda), used: True
TPU available: False, 

Fold 1: accuracy=0.8585  roc_auc=0.9146  (fit 5765.5s)


...

Fold 2: accuracy=0.8593  roc_auc=0.9141  (fit 5917.4s)


/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/uti

Fold 3: accuracy=0.8598  roc_auc=0.9152  (fit 6123.3s)


/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
GPU available: True (cuda), used: True
TPU available: False, 

Fold 4: accuracy=0.8589  roc_auc=0.9127  (fit 6322.9s)

OOF accuracy: 0.8590
OOF ROC-AUC:  0.9140
Folds:        0.8590 ± 0.0005

Run complete. Call save_experiment(result) to log this run permanently.


In [12]:
# Step 2 — Save the run (optional). Review the OOF ROC-AUC printed above first, and
# only save if the fold-contract cell passed.
run_id = save_experiment(result)

Saved to: /kaggle/working/Predict-Customer-Churn/experiments/runs/20260612-185205-91e530


### Build a submission (optional)

`test_proba_mean` is the fold-bagged (5 folds × 2 seeds) churn probability for the full
test set. The competition metric is ROC-AUC, so submit the probability directly.

In [13]:
# submission = pd.DataFrame({
#     'id':    test_df['id'],
#     'Churn': result['artifacts']['test_proba_mean'],
# })
# submission.to_csv('/kaggle/working/submission.csv', index=False)
# print(submission.head())
# print('wrote /kaggle/working/submission.csv', submission.shape)

### Bundle run artifacts into one zip

Zips the run directory and `runs.csv` into a single archive on the Output tab. To fold
the run back into the local repo, follow **§7–8 of `docs/kaggle_gpu_workflow.md`**, then
rerun `scripts/check_oof_alignment.py` with this run added as the post-merge gate.

In [14]:
import shutil
from pathlib import Path
from src.tracking import RUNS_DIR, RUNS_CSV

BUNDLE = Path('/kaggle/working/bundle')
if BUNDLE.exists():
    shutil.rmtree(BUNDLE)

# 1) heavy run artifacts (params, oof_proba, test_proba_*, metrics, env, git diff)
shutil.copytree(RUNS_DIR / run_id, BUNDLE / 'runs' / run_id)
# 2) the master index row
shutil.copy(RUNS_CSV, BUNDLE / 'runs.csv')

archive = shutil.make_archive(f'/kaggle/working/{run_id}_bundle', 'zip', BUNDLE)
print('wrote', archive)

wrote /kaggle/working/20260612-185205-91e530_bundle.zip
